In [3]:
!git clone https://github.com/aoteromiguens/app_scoring.git  # Clonamos el repositorio de GitHub al disco de Colab
%cd app_scoring # Le decimos a Colab que se meta dentro de la carpeta que acaba de descargar

import pandas as pd
import numpy as np
from joblib import load # Necesario para cargar los archivos .joblib
import json # Para una salida formateada en el ejemplo
import ipywidgets as widgets # Para crear la interfaz de usuario interactiva
from IPython.display import display, clear_output # Para mostrar widgets y limpiar la salida
import matplotlib.pyplot as plt # Para gráficos
import matplotlib.patches as mpatches # Para el medidor/termómetro

# --- 1. CONFIGURACIÓN INICIAL DE UMBRALES ---
umbrales_riesgo = {
    'BAJO': 0.7,  # Score < 0.7 es BAJO
    'MEDIO': 1.5  # Score >= 0.7 y < 1.5 es MEDIO; Score >= 1.5 es ALTO
}
print("Umbrales de riesgo definidos:")
print(f"  Bajo Riesgo: < {umbrales_riesgo['BAJO']}")
print(f"  Medio Riesgo: {umbrales_riesgo['BAJO']} a < {umbrales_riesgo['MEDIO']}")
print(f"  Alto Riesgo: >= {umbrales_riesgo['MEDIO']}")

# --- 2. Carga de Modelos ---
print("\n--- Cargando modelos desde archivos .joblib ---")
try:
      %cd app_scoring
      print("Cargando Scaler...")
      scalerL = load('models/mi_scaler_limpio.joblib')
      print("✅ Scaler OK")

      print("Cargando DBSCAN...")
      dbscan_classifierL = load('models/dbscan_classifier_for_inference.joblib')
      print("✅ DBSCAN OK")

      print("Cargando KMeans 2...")
      kmeans_k2_modelL = load('models/modelo_kmeans_k2.joblib')
      print("✅ KMeans 2 OK")

      print("Cargando KMeans 3...")
      kmeans_k3_modelL = load('models/modelo_kmeans_k3.joblib')
      print("✅ KMeans 3 OK")

      print("Cargando Random Forest...")
      rf_modelL = load('models/random_forest_model_base.joblib')
      print("✅ Random Forest OK")

      print("✅ Todos los modelos cargados exitosamente.")

except FileNotFoundError as e:
    print(f"❌ ERROR: Uno o más archivos .joblib no se encontraron: {e}")
    print("Asegúrate de que todos los archivos (mi_scaler_limpio.joblib, random_forest_model_base.joblib, modelo_kmeans_k2.joblib, modelo_kmeans_k3.joblib, dbscan_classifier_for_inference.joblib) estén subidos en la raíz de tu entorno de Colab.")
    exit()
except Exception as e:
    print(f"❌ Ocurrió un error al cargar los modelos: {e}")
    exit()


# --- 3. Configuración de Pesos de Ponderación y Riesgo Promedio de K-Means ---
print("\n--- Verificando y configurando los valores para el Score Híbrido ---")

# PESOS DE PONDERACIÓN:
peso_rf = 0.5
peso_k3 = 0.3
peso_k2 = 0.2

print("\nPesos de Ponderación para el Score Híbrido:")
print(f"  Peso Random Forest (peso_rf): {peso_rf}")
print(f"  Peso K-Means (k=3) (peso_k3): {peso_k3}")
print(f"  Peso K-Means (k=2) (peso_k2): {peso_k2}")
print(f"  Suma de Pesos: {peso_rf + peso_k3 + peso_k2}")


# RIESGO PROMEDIO POR CLÚSTER K-MEANS:
kmeans_k2_avg_risk = {
    0: 0.585826, # Promedio de Riesgo_num para el Cluster 0 de K-Means k=2
    1: 1.559824  # Promedio de Riesgo_num para el Cluster 1 de K-Means k=2
}

kmeans_k3_avg_risk = {
    0: 0.843419, # Promedio de Riesgo_num para el Cluster 0 de K-Means k=3
    1: 1.582101, # Promedio de Riesgo_num para el Cluster 1 de K-Means k=3
    2: 0.192106  # Promedio de Riesgo_num para el Cluster 2 de K-Means k=3
}

print("\nPromedio de Riesgo ('Riesgo_num') por Clúster K-Means:")
print("  K-Means (k=2):")
for cluster_id, avg_risk in kmeans_k2_avg_risk.items():
    print(f"    Cluster {cluster_id}: {avg_risk}")

print("  K-Means (k=3):")
for cluster_id, avg_risk in kmeans_k3_avg_risk.items():
    print(f"    Cluster {cluster_id}: {avg_risk}")

print("\n--- Modelos y configuración listos para la inferencia ---")


# --- 4. Definición de Características (asegúrate que coincidan con tu entrenamiento) ---
# Ahora 'Product_Density' ya NO es una característica de entrada directa, se calcula.
features_for_model_CLEAN = [
    'Age', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card',
    'Num_of_Loan_numeric', 'Interest_Rate', 'Delay_from_due_date',
    'Num_of_Delayed_Payment', 'Changed_Credit_Limit_numeric',
    'Num_Credit_Inquiries', 'Credit_Mix_numeric',
    'Credit_Utilization_Ratio', 'Payment_of_Min_Amount',
    'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance',
    'Credit_History_Age_Months', 'EMI_to_Income', 'Product_Density', # ¡Aquí está!
    'Spent_Level', 'Value_Level'
]

top_10_features = [
    'Monthly_Inhand_Salary',
    'Product_Density', # ¡Aquí también!
    'Monthly_Balance',
    'Interest_Rate',
    'Total_EMI_per_month',
    'Credit_History_Age_Months',
    'Amount_invested_monthly',
    'EMI_to_Income',
    'Changed_Credit_Limit_numeric',
    'Num_Credit_Inquiries'
]


# --- 5. Función de Inferencia para un Cliente Nuevo ---
def predict_client_risk(client_data: dict):
    """
    Toma los datos de un cliente nuevo (ahora incluyendo 'Total_Products')
    y devuelve su score de riesgo y nivel predicho.
    """
    df_single_client = pd.DataFrame([client_data])

    # Calcular EMI_to_Income (manejar división por cero)
    if df_single_client['Monthly_Inhand_Salary'].iloc[0] != 0:
        df_single_client['EMI_to_Income'] = df_single_client['Total_EMI_per_month'] / df_single_client['Monthly_Inhand_Salary']
    else:
        df_single_client['EMI_to_Income'] = 0

    # --- NUEVO CÁLCULO DE PRODUCT_DENSITY ---
    # Asegúrate de que Credit_History_Age_Months siempre esté disponible (es una entrada del usuario)
    total_products_val = df_single_client['Total_Products'].iloc[0]
    credit_history_age_months_val = df_single_client['Credit_History_Age_Months'].iloc[0]

    # Calcular Product_Density según la fórmula proporcionada
    # Evitar división por cero si Credit_History_Age_Months es -1 o muy bajo, sumando 1
    if (credit_history_age_months_val + 1) != 0:
        df_single_client['Product_Density'] = total_products_val / (credit_history_age_months_val + 1)
    else:
        df_single_client['Product_Density'] = 0 # O un valor por defecto apropiado

    # Eliminar 'Total_Products' ya que no es una característica para el modelo
    df_single_client = df_single_client.drop(columns=['Total_Products'])


    # Rellenar con 0 las características no proporcionadas directamente por el usuario
    for col in features_for_model_CLEAN:
        if col not in df_single_client.columns:
            df_single_client[col] = 0 # Valor por defecto. ¡Ajusta si usaste la media, moda, etc.!

    df_single_client_ordered = df_single_client[features_for_model_CLEAN]

    X_scaled_all_features_client = scalerL.transform(df_single_client_ordered)
    X_scaled_df_client = pd.DataFrame(X_scaled_all_features_client, columns=features_for_model_CLEAN)
    X_clust_client = X_scaled_df_client[top_10_features].values

    riesgo_rf_predicho = rf_modelL.predict(X_scaled_all_features_client)[0]
    cluster_k2 = kmeans_k2_modelL.predict(X_clust_client)[0]
    cluster_k3 = kmeans_k3_modelL.predict(X_clust_client)[0]
    cluster_dbscan = dbscan_classifierL.predict(X_clust_client)[0]

    score_temporal = 0
    nivel_riesgo_final = ""

    # Lógica de priorización de DBSCAN
    if cluster_dbscan == 0: # Alto Riesgo Extremo
        score_temporal = 2.0
        nivel_riesgo_final = "ALTO EXTREMO (DBSCAN Cluster 0)"
    elif cluster_dbscan == 1: # Bajo Riesgo Premium
        score_temporal = 0.1
        nivel_riesgo_final = "BAJO PREMIUM (DBSCAN Cluster 1)"
    elif cluster_dbscan == 2: # Moderado-Específico
        score_temporal = 1.0
        nivel_riesgo_final = "MEDIO (DBSCAN Cluster 2)"
    else: # Si es ruido (-1) o cluster no mapeado
        riesgo_k2_asociado = kmeans_k2_avg_risk.get(cluster_k2, 0)
        riesgo_k3_asociado = kmeans_k3_avg_risk.get(cluster_k3, 0)

        score_temporal = (peso_rf * riesgo_rf_predicho) + \
                         (peso_k3 * riesgo_k3_asociado) + \
                         (peso_k2 * riesgo_k2_asociado)

        if score_temporal < umbrales_riesgo['BAJO']:
            nivel_riesgo_final = "BAJO (Ponderado)"
        elif score_temporal < umbrales_riesgo['MEDIO']:
            nivel_riesgo_final = "MEDIO (Ponderado)"
        else:
            nivel_riesgo_final = "ALTO (Ponderado)"

    score_temporal = round(float(score_temporal), 2)

    response = {
        "score_temporal": score_temporal,
        "Nivel_Riesgo_Final": nivel_riesgo_final,
        "detalles_modelos": {
            "Riesgo_RF_Predicho": float(riesgo_rf_predicho),
            "Cluster_KMeans_k2": int(cluster_k2),
            "Riesgo_KMeans_k2_Asociado": float(kmeans_k2_avg_risk.get(cluster_k2, 'N/A')),
            "Cluster_KMeans_k3": int(cluster_k3),
            "Riesgo_KMeans_k3_Asociado": float(kmeans_k3_avg_risk.get(cluster_k3, 'N/A')),
            "Cluster_DBSCAN": int(cluster_dbscan),
            "Product_Density_Calculada": float(df_single_client['Product_Density'].iloc[0]) # Para verificar
        }
    }
    return response

# --- 6. Función para Dibujar el Medidor de Riesgo (Termómetro) ---
def plot_risk_meter(score, risk_level):
    fig, ax = plt.subplots(figsize=(8, 1.5))
    ax.set_xlim(0, 2.0) # El rango de tu score_temporal
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_xticks(np.arange(0, 2.1, 0.5))
    ax.set_xticklabels([0, 0.5, 1.0, 1.5, 2.0])

    # Rangos de riesgo (colores basados en tus umbrales y la lógica de DBSCAN)
    # BAJO: < 0.7
    # MEDIO: 0.7 a < 1.5
    # ALTO: >= 1.5
    # ALTO EXTREMO (DBSCAN 0): 2.0
    # BAJO PREMIUM (DBSCAN 1): 0.1
    # MEDIO (DBSCAN 2): 1.0

    # Colores base para los rangos generales
    ax.axvspan(0, umbrales_riesgo['BAJO'], color='#86E3CE', alpha=0.7, label='Bajo Riesgo (<0.7)') # Verde claro
    ax.axvspan(umbrales_riesgo['BAJO'], umbrales_riesgo['MEDIO'], color='#FFDD73', alpha=0.7, label='Medio Riesgo (0.7-1.5)') # Amarillo
    ax.axvspan(umbrales_riesgo['MEDIO'], 2.0, color='#FF6663', alpha=0.7, label='Alto Riesgo (>=1.5)') # Rojo claro

    # Marcar los puntos de riesgo específicos de DBSCAN (si caen dentro del rango)
    if 'ALTO EXTREMO (DBSCAN Cluster 0)' in risk_level:
        ax.axvspan(1.9, 2.1, color='#9C0000', alpha=0.9, label='DBSCAN Alto Extremo (2.0)') # Rojo oscuro
    elif 'BAJO PREMIUM (DBSCAN Cluster 1)' in risk_level:
        ax.axvspan(0, 0.2, color='#28A745', alpha=0.9, label='DBSCAN Bajo Premium (0.1)') # Verde oscuro
    elif 'MEDIO (DBSCAN Cluster 2)' in risk_level:
        ax.axvspan(0.9, 1.1, color='#FFBB00', alpha=0.9, label='DBSCAN Medio (1.0)') # Naranja

    # Colocar el indicador del score actual
    ax.plot([score, score], [0, 1], color='black', linewidth=3, linestyle='--', label=f'Score Actual: {score}')
    ax.scatter(score, 0.5, color='black', s=200, marker='D', zorder=5) # Diamante para el score

    ax.set_title(f"Nivel de Riesgo del Cliente: {risk_level}", fontsize=14)
    ax.set_xlabel("Score de Riesgo", fontsize=12)
    ax.grid(axis='x', linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()


# --- 7. Interfaz de Usuario con ipywidgets ---

# Definir los widgets para las 9 características principales + Total_Products
widgets_input = {
    'Monthly_Inhand_Salary': widgets.FloatText(description='Salario Mensual:', value=50000, min=0, step=1000),
    'Total_EMI_per_month': widgets.FloatText(description='EMI Total Mensual:', value=15000, min=0, step=500),
    'Credit_History_Age_Months': widgets.IntText(description='Historial Crédito (Meses):', value=80, min=0),
    'Total_Products': widgets.IntText(description='Cantidad de Productos:', value=5, min=0), # NUEVO WIDGET
    'Monthly_Balance': widgets.FloatText(description='Balance Mensual:', value=25000, min=0, step=1000),
    'Interest_Rate': widgets.FloatText(description='Tasa Interés (%):', value=10.0, min=0.0, step=0.1),
    'Amount_invested_monthly': widgets.FloatText(description='Monto Invertido Mensual:', value=6000, min=0, step=100),
    'Changed_Credit_Limit_numeric': widgets.FloatText(description='Cambio Límite Crédito:', value=0, step=100),
    'Num_Credit_Inquiries': widgets.IntText(description='Consultas Crédito (Últ. Año):', value=2, min=0),
}

# Botón para ejecutar la predicción
button_predict = widgets.Button(description="Calcular Riesgo")
output_area = widgets.Output() # Área para mostrar los resultados

# Función que se ejecuta al presionar el botón
def on_predict_button_clicked(b):
    with output_area:
        clear_output(wait=True) # Limpiar salida anterior
        user_data = {name: widget.value for name, widget in widgets_input.items()}

        # Añadir valores por defecto para las otras 11 características no ingresadas por el usuario
        # ¡Asegúrate de que esta estrategia de relleno coincide con tu preprocesamiento original!
        user_data['Age'] = 35
        user_data['Num_Bank_Accounts'] = 3
        user_data['Num_Credit_Card'] = 2
        user_data['Num_of_Loan_numeric'] = 1
        user_data['Delay_from_due_date'] = 0
        user_data['Num_of_Delayed_Payment'] = 0
        user_data['Credit_Mix_numeric'] = 1 # 0: Bad, 1: Standard, 2: Good (asumiendo esta codificación)
        user_data['Credit_Utilization_Ratio'] = 0.4
        user_data['Payment_of_Min_Amount'] = 0 # 0: No, 1: Yes (asumiendo esta codificación)
        user_data['Spent_Level'] = 1 # 0: Bajo, 1: Medio, 2: Alto (asumiendo esta codificación)
        user_data['Value_Level'] = 1 # 0: Bajo, 1: Medio, 2: Alto (asumiendo esta codificación)

        # Ejecutar la predicción
        try:
            results = predict_client_risk(user_data)

            # Mostrar resultados textuales
            print("--- Resultados del Análisis de Riesgo ---")
            print(f"Score Temporal: {results['score_temporal']}")
            print(f"Nivel de Riesgo Final: {results['Nivel_Riesgo_Final']}")
            print("\nDetalles de los Modelos:")
            print(f"  Riesgo Random Forest Predicho: {results['detalles_modelos']['Riesgo_RF_Predicho']}")
            print(f"  K-Means (k=2) Clúster: {results['detalles_modelos']['Cluster_KMeans_k2']} (Riesgo Asociado: {results['detalles_modelos']['Riesgo_KMeans_k2_Asociado']})")
            print(f"  K-Means (k=3) Clúster: {results['detalles_modelos']['Cluster_KMeans_k3']} (Riesgo Asociado: {results['detalles_modelos']['Riesgo_KMeans_k3_Asociado']})")
            print(f"  DBSCAN Clúster: {results['detalles_modelos']['Cluster_DBSCAN']}")
            print(f"  Product_Density Calculada: {results['detalles_modelos']['Product_Density_Calculada']:.2f}")


            # Mostrar gráfico de termómetro de riesgo
            plot_risk_meter(results['score_temporal'], results['Nivel_Riesgo_Final'])

        except Exception as e:
            print(f"❌ Error al calcular el riesgo: {e}")
            print("Asegúrate de que los valores de entrada son válidos y los modelos están cargados.")

# Asignar la función al botón
button_predict.on_click(on_predict_button_clicked)

# Organizar los widgets en una interfaz
input_widgets_layout = widgets.VBox([
    widgets.HBox([widgets_input['Monthly_Inhand_Salary'], widgets_input['Total_EMI_per_month']]),
    widgets.HBox([widgets_input['Credit_History_Age_Months'], widgets_input['Total_Products']]), # NUEVO EN LA INTERFAZ
    widgets.HBox([widgets_input['Monthly_Balance'], widgets_input['Interest_Rate']]),
    widgets.HBox([widgets_input['Amount_invested_monthly'], widgets_input['Changed_Credit_Limit_numeric']]),
    widgets.HBox([widgets_input['Num_Credit_Inquiries']]),
    button_predict
])

# Mostrar la interfaz
print("--- Interfaz de Predicción de Riesgo ---")
display(input_widgets_layout, output_area)
print("\n¡Listo para la demo! Ingresa los datos del cliente y haz clic en 'Calcular Riesgo'.")

Cloning into 'app_scoring'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 25 (delta 2), reused 4 (delta 2), pack-reused 19 (from 1)
Receiving objects: 100% (25/25), 42.65 MiB | 24.17 MiB/s, done.
Resolving deltas: 100% (6/6), done.
[Errno 2] No such file or directory: 'app_scoring # Le decimos a Colab que se meta dentro de la carpeta que acaba de descargar'
/content/app_scoring/app_scoring
Umbrales de riesgo definidos:
  Bajo Riesgo: < 0.7
  Medio Riesgo: 0.7 a < 1.5
  Alto Riesgo: >= 1.5

--- Cargando modelos desde archivos .joblib ---
/content/app_scoring/app_scoring/app_scoring
Cargando Scaler...
✅ Scaler OK
Cargando DBSCAN...
✅ DBSCAN OK
Cargando KMeans 2...
✅ KMeans 2 OK
Cargando KMeans 3...
✅ KMeans 3 OK
Cargando Random Forest...
✅ Random Forest OK
✅ Todos los modelos cargados exitosamente.

--- Verificando y configurando los valores para el Score Híbrido ---

Pesos de Ponderación 

Output()


¡Listo para la demo! Ingresa los datos del cliente y haz clic en 'Calcular Riesgo'.
